In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point
import gc
import math
import matplotlib.figure as figure
import matplotlib.animation as animation
from shapely.geometry import LineString
from datetime import datetime, timedelta
from shapely.wkt import loads
import sys
import gc

In [131]:
#create 24 points per line

# Load the original GeoDataFrame 
gdf = gpd.read_file('C:\\Users\\andre\\OneDrive\\Dokumente\\UNI\\Master_Digital_Humanities\\S3_WS2024\\LP_Data_Analysis_Project\\data\\stdb_lines.gpkg')

# Ensure that YEARDEP is converted to a numeric value if it's not already
gdf['YEARDEP'] = pd.to_numeric(gdf['YEARDEP'], errors='coerce')

# Handle missing or invalid DATEDEP
gdf['DATEDEP'] = pd.to_datetime(gdf['DATEDEP'], errors='coerce')  # Convert to datetime, set invalid entries as NaT

# If DATEDEP is missing, use YEARDEP to generate a date with 01.01.
gdf.loc[gdf['DATEDEP'].isna(), 'DATEDEP'] = gdf.loc[gdf['DATEDEP'].isna(), 'YEARDEP'].apply(
    lambda x: datetime(year=int(x), month=6, day=1) if pd.notna(x) else pd.NaT
)

# Ensure DATEDEP is in datetime format (handle any previous errors)
gdf['DATEDEP'] = pd.to_datetime(gdf['DATEDEP'], errors='coerce')

# Filter out rows where the year in DATEDEP is before 1600
gdf = gdf[gdf['DATEDEP'].dt.year >= 1600]

start_date = datetime(1600, 1, 1)
end_date = datetime(1880, 1, 1)

# Create a list of all month-start dates between start_date and end_date
current_date = start_date
date_range = []
while current_date < end_date:
    date_range.append(current_date)
    # Increment by one month
    if current_date.month == 12:
        current_date = datetime(current_date.year + 1, 1, 1)
    else:
        current_date = datetime(current_date.year, current_date.month + 1, 1)

# Create the dictionary to map dates to vertex indices
date_to_index = {date.strftime("%Y-%m"): i + 1 for i, date in enumerate(date_range)}

# Convert the date_to_index dictionary into a DataFrame for joining
date_index_df = pd.DataFrame.from_dict(date_to_index, orient='index', columns=['vertex_index']).reset_index()
date_index_df.rename(columns={'index': 'DATE'}, inplace=True)

# Function to generate points along a line (now with 24 points instead of 12)
def generate_points_along_line(line, num_points=24):
    # Get the length of the line
    line_length = line.length
    # Generate equally spaced points along the line
    points = [line.interpolate(i * line_length / (num_points - 1)) for i in range(num_points)]
    return points

# List to store points data
points_data = []

# Iterate over each row in the GeoDataFrame
for index, row in gdf.iterrows():
    # Generate 24 points along the line (double the previous amount)
    points = generate_points_along_line(row['geometry'], num_points=24)
    
    # Extract the year and month from DATEDEP and set the starting date to the first of the month
    departure_date = row['DATEDEP']
    year = departure_date.year
    month = departure_date.month
    starting_date = datetime(year, month, 1)
    starting_date_str = starting_date.strftime("%Y-%m")  # Format as "YYYY-MM"
    
    # Check if the starting date exists in date_to_index
    if starting_date_str not in date_to_index:
        continue  # Skip if the date is out of range

    # Retrieve the vertex index for the starting date
    base_index = date_to_index[starting_date_str]
    
    # Repeat start point 3 times
    for i in range(-6, 0):  # Repeat for 3 frames
        points_data.append({
            "point_id": row['VOYAGEID'],
            "YEARDEP": row['YEARDEP'],
            "DATE": starting_date_str,
            "vertex_index": base_index + i,
            "geometry": points[0],  # Add the first point as the geometry for start point
            "NATINIMP": row['NATINIMP'],  # Include the NATINIMP information
            "SLAMIMP": row['SLAMIMP']   # Include the SLAMIMP information
        })
    
    # Generate the 24 monthly points for the current voyage (2 frames per month)
    for month_offset in range(24):  # Now 12 months
        # Calculate the current month
        current_date = starting_date + pd.DateOffset(months=month_offset)
        current_date_str = current_date.strftime("%Y-%m")
        
        # Check if the current date exists in date_to_index
        if current_date_str not in date_to_index:
            continue
        
        # Append the point data
        points_data.append({
            "point_id": row['VOYAGEID'],
            "YEARDEP": row['YEARDEP'],
            "DATE": current_date_str,
            "vertex_index": date_to_index[current_date_str],
            "geometry": points[month_offset],  # Assign the geometry based on the month offset
            "NATINIMP": row['NATINIMP'],  # Include the NATINIMP information
            "SLAMIMP": row['SLAMIMP'] 
        })
    
    # Repeat end point 3 times
    for i in range(0,6):  # Repeat for 3 frames
        points_data.append({
            "point_id": row['VOYAGEID'],
            "YEARDEP": row['YEARDEP'],
            "DATE": current_date_str,  # Use the last valid date
            "vertex_index": date_to_index[current_date_str] + i,
            "geometry": points[-1],  # Add the last point as the geometry for the end point
            "NATINIMP": row['NATINIMP'],  # Include the NATINIMP information
            "SLAMIMP": row['SLAMIMP']   # Include the SLAMIMP information
        })

# Create a new GeoDataFrame with the points data
points_df = gpd.GeoDataFrame(points_data, geometry='geometry')

# Save the final DataFrame to a file
#points_df.to_file('C:\\Users\\andre\\OneDrive\\Dokumente\\UNI\\Master_Digital_Humanities\\S3_WS2024\\LP_Data_Analysis_Project\\data\\points_with_date_indices_and_attributes.gpkg', driver="GPKG")

# save as csv
points_df.to_csv('C:\\Users\\andre\\OneDrive\\Dokumente\\UNI\\Master_Digital_Humanities\\S3_WS2024\\LP_Data_Analysis_Project\\data\\points_with_date_indices24.csv', index=False)

C:\Users\andre\AppData\Local\Temp\ipykernel_4460\1571425959.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[datetime.datetime(1817, 6, 1, 0, 0) datetime.datetime(1817, 6, 1, 0, 0)
 datetime.datetime(1817, 6, 1, 0, 0) ...
 datetime.datetime(1850, 6, 1, 0, 0) datetime.datetime(1851, 6, 1, 0, 0)
 datetime.datetime(1851, 6, 1, 0, 0)]' has dtype incompatible with datetime64[ms], please explicitly cast to a compatible dtype first.
  gdf.loc[gdf['DATEDEP'].isna(), 'DATEDEP'] = gdf.loc[gdf['DATEDEP'].isna(), 'YEARDEP'].apply(


In [2]:
# Load the points CSV into a GeoDataFrame
points_gdf_all_origin = gpd.read_file('C:\\Users\\andre\\OneDrive\\Dokumente\\UNI\\Master_Digital_Humanities\\S3_WS2024\\LP_Data_Analysis_Project\\data\\points_with_date_indices24.csv'
                           ,encoding='utf-8')

# Load Natural Earth data from local path
natural_earth_path = 'C:\\Users\\andre\\OneDrive\\Dokumente\\UNI\\Master_Digital_Humanities\\S3_WS2024\\LP_Data_Analysis_Project\\data\\basemap\\ne_50m_land\\ne_50m_land.shp'
river_path = 'C:\\Users\\andre\\OneDrive\\Dokumente\\UNI\\Master_Digital_Humanities\\S3_WS2024\\LP_Data_Analysis_Project\\data\\basemap\\ne_50m_rivers_lake_centerlines\\ne_50m_rivers_lake_centerlines.shp'  # Land data
ocean_path = 'C:\\Users\\andre\\OneDrive\\Dokumente\\UNI\\Master_Digital_Humanities\\S3_WS2024\\LP_Data_Analysis_Project\\data\\basemap\\ne_50m_ocean\\ne_50m_ocean.shp'  # Ocean data

# Load the shapefiles
land = gpd.read_file(natural_earth_path)
ocean = gpd.read_file(ocean_path)
rivers = gpd.read_file(river_path)

In [3]:
# Define the color mapping for NATINIMP
color_mapping = {
    'Denmark / Baltic': 'wheat',
    'France': 'navy',
    "Great Britain": "firebrick",
    "Netherlands": "darkorange",
    'Portugal / Brazil': 'limegreen',
    'Spain / Uruguay': 'olivedrab',
    'USA': 'darkorchid',
    "Other": "grey"
}

In [4]:
#create animation from points

points_gdf_all_origin = points_gdf_all_origin[points_gdf_all_origin["geometry"]!='']

# Ensure the geometry column contains valid Shapely geometries
points_gdf_all_origin['geometry'] = points_gdf_all_origin['geometry'].apply(lambda x: loads(x) if isinstance(x, str) else x)

# Ensure it's a GeoDataFrame by converting it
points_gdf_all = gpd.GeoDataFrame(
    points_gdf_all_origin,
    geometry=points_gdf_all_origin['geometry'],  # Use the geometry column directly
    crs="EPSG:4326"  # Set the CRS (assuming WGS84)
)

points_gdf_all.set_crs("EPSG:4326", inplace=True, allow_override=True)

points_gdf_all['color'] = points_gdf_all['NATINIMP'].map(color_mapping).fillna('grey')  # Default to grey if no match

# Convert the 'vertex_index' column to numeric (coerce errors to NaN if non-numeric values are present)
points_gdf_all['vertex_index'] = pd.to_numeric(points_gdf_all['vertex_index'], errors='coerce')

points_gdf_all['SLAMIMP'] = pd.to_numeric(points_gdf_all['SLAMIMP'], errors='coerce')

#points_gdf_all = points_gdf_all.to_crs("ESRI:53032")

# Define the min and max marker size
min_marker_size = 5
max_marker_size = 40

# Normalize SLAMIMP to be between 5 and 20
min_slamimp = points_gdf_all['SLAMIMP'].min()
max_slamimp = points_gdf_all['SLAMIMP'].max()

# Use linear scaling to adjust marker sizes
points_gdf_all['marker_size'] = min_marker_size + (points_gdf_all['SLAMIMP'] - min_slamimp) * (max_marker_size - min_marker_size) / (max_slamimp - min_slamimp)

# Filter the DataFrame for the Americas (bounding box)
americas_bbox = land.cx[-100:0, -30:40]

# Sort the unique vertex_index values (now as numbers)
sorted_vertex_indices = sorted(points_gdf_all['vertex_index'].unique())

# Loop through all unique vertex_index values in ascending order
#for idx, vertex_value in enumerate(sorted_vertex_indices):
for idx, vertex_value in enumerate(sorted_vertex_indices):
    vertex_value = sorted_vertex_indices[idx]
    # Filter data for current vertex_index
    points_gdf = points_gdf_all[points_gdf_all['vertex_index'] == vertex_value]

    # Plot the map
    #fig, ax = plt.subplots(figsize=(12, 8))

    fig = figure.Figure(figsize=(12, 8))
    ax = fig.subplots(1, 1)

    # Plot Natural Earth basemap, focusing on Americas
    americas = land.cx[-100:0, -30:40]  # Specify bounds for Americas
    americas.plot(ax=ax, color='#fff5e1', edgecolor='black')

    # Plot the ocean layer (use your local ocean shapefile path)
    ocean.plot(ax=ax, color='#d3f2ff', edgecolor='none', label='Ocean')

    # Plot the rivers layer (use your local rivers shapefile path)
    rivers.plot(ax=ax, color='#4f7bfe', edgecolor='none', label='Rivers', alpha=0.6, linewidth=0.4)

    # Overlay the points, coloring them based on NATINIMP
    for natinimp, group in points_gdf.groupby('NATINIMP'):
        if natinimp != "nan":
            group.plot(
                ax=ax,
                color=group['color'].iloc[0],  # Use the mapped color
                markersize=round(group['marker_size'],0),
                edgecolor='black',  # Black border for all points
                linewidth=0.4,  # Thickness of the border
                label=natinimp,
                alpha=0.9
            )

    # Add YEARAM text in the bottom-right corner (from the first point)
    year_value = math.floor(1678+((idx-6)/12))
    ax.text(0.98, 0.02, f'{year_value}', transform=ax.transAxes, ha='right', va='bottom', fontsize=12, color='black',
            bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.5'))

    # Hide the axis ticks and labels
    ax.set_xticks([])
    ax.set_yticks([])

    # Customize the map
    ax.set_xlim([-100, 60])  # Focus on the Americas
    ax.set_ylim([-40, 40])

    ax.set_aspect('auto')

    # Save the map as an image file
    path = 'D:\\routes_animation2_12'
    fig.savefig(path+f'\\routes2_{idx + 1}.png', dpi=300, bbox_inches='tight')  # Save with filename "routes2_X.png"

    sys.stdout.write(f"\rProcessed {idx + 1}/{len(points_gdf_all['vertex_index'].unique())} vertex_index values")
    sys.stdout.flush()

    del points_gdf
    del fig, ax

    gc.collect()

print("Execution finished")


Processed 2297/2297 vertex_index valuesExecution finished
